In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import *
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from datetime import datetime

LANDING_PATH    = "/Volumes/nyctaxi/landing/yellow_taxi/regions/"
TARGET_TABLE    = "NYCTAXI.SILVER.REGIONS_SCD_2"
WATERMARK_TABLE = "NYCTAXI.SILVER.REGIONS_FILE_WATERMARK"

# ── ID is uppercase to match Delta table schema ────────────────────────────────
scd2_schema = StructType([
    StructField("ID",                   IntegerType(),  True),
    StructField("name",                 StringType(),   True),
    StructField("effective_start_date", TimestampType(), True),
    StructField("effective_end_date",   TimestampType(), True),
    StructField("load_timestamp",       TimestampType(), True),
])

# ── Discover all CSV files with metadata ──────────────────────────────────────
all_files = {
    f.path: {"size": f.size, "modificationTime": f.modificationTime}
    for f in dbutils.fs.ls(LANDING_PATH)
    if f.path.endswith(".csv")
}

# ── Get watermark records ─────────────────────────────────────────────────────
watermark_records = {
    row.file_name: {"size": row.file_size, "modificationTime": row.file_modified_time}
    for row in spark.table(WATERMARK_TABLE).collect()
}

# ── Classify files ────────────────────────────────────────────────────────────
new_files      = []
modified_files = []

for file_path, meta in all_files.items():
    if file_path not in watermark_records:
        new_files.append(file_path)
    elif (
        meta["size"]             != watermark_records[file_path]["size"] or
        meta["modificationTime"] != watermark_records[file_path]["modificationTime"]
    ):
        modified_files.append(file_path)

files_to_process = sorted(new_files + modified_files)

print(f"New files      : {sorted(new_files)}")
print(f"Modified files : {sorted(modified_files)}")

if not files_to_process:
    print("No new or modified files to process.")
else:
    print(f"\nProcessing {len(files_to_process)} file(s)...")

    for file_path in files_to_process:
        print(f"\nProcessing: {file_path}")

        now = datetime.utcnow()

        # ── Read incoming file ─────────────────────────────────────────────────
        updates_df = (
            spark.read.format("csv")
            .option("header", "true")
            .schema("ID INT, name STRING, date STRING")  # ID uppercase
            .load(file_path)
            .withColumn("effective_start_date", to_timestamp(col("date"), "dd-MM-yyyy"))
            .drop("date")
        )

        # Collect to Python tuples immediately
        incoming_rows = [
            (
                row.ID,
                row.name,
                row.effective_start_date,
                None,
                now
            )
            for row in updates_df.collect()
        ]

        row_count = spark.table(TARGET_TABLE).count()

        if row_count == 0:
            # ── Initial load ───────────────────────────────────────────────────
            initial_df = spark.createDataFrame(incoming_rows, schema=scd2_schema)
            initial_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
            print(f"Initial load complete: {file_path}")

        else:
            # ── Snapshot active records BEFORE any changes ─────────────────────
            existing_active_map = {
                row.ID: row.name
                for row in spark.table(TARGET_TABLE)
                .filter(col("effective_end_date").isNull())
                .collect()
            }
            existing_all_ids = {
                row.ID
                for row in spark.table(TARGET_TABLE)
                .select("ID").distinct()
                .collect()
            }

            # ── Classify incoming records ──────────────────────────────────────
            changed_tuples = []
            new_id_tuples  = []

            for (ID, name, eff_start, eff_end, load_ts) in incoming_rows:
                if ID not in existing_all_ids:
                    print(f"  New ID    : ID={ID}, name={name}")
                    new_id_tuples.append((ID, name, eff_start, None, now))
                elif existing_active_map.get(ID) != name:
                    print(f"  Changed   : ID={ID}, {existing_active_map.get(ID)} → {name}")
                    changed_tuples.append((ID, name, eff_start, None, now))
                else:
                    print(f"  Unchanged : ID={ID}, name={name} — skipped")

            changed_count = len(changed_tuples)
            new_count     = len(new_id_tuples)
            print(f"\nChanged: {changed_count} | New IDs: {new_count}")

            # ── Expire old records ─────────────────────────────────────────────
            if changed_count > 0:
                changed_df = spark.createDataFrame(changed_tuples, schema=scd2_schema)
                target = DeltaTable.forName(spark, TARGET_TABLE)
                target.alias("t").merge(
                    changed_df.alias("s"),
                    """
                    t.ID = s.ID
                    AND t.effective_end_date IS NULL
                    AND t.name != s.name
                    """
                ).whenMatchedUpdate(set={
                    "effective_end_date" : col("s.effective_start_date"),
                    "load_timestamp"     : current_timestamp()
                }).execute()
                print(f"Expired {changed_count} record(s)")

            # ── Insert new versions + new IDs ──────────────────────────────────
            rows_to_insert = changed_tuples + new_id_tuples

            if rows_to_insert:
                insert_df = spark.createDataFrame(rows_to_insert, schema=scd2_schema)
                insert_df.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
                print(f"Inserted {len(rows_to_insert)} record(s)")
            else:
                print("No changes detected — skipping")

        # ── Update watermark ───────────────────────────────────────────────────
        spark.createDataFrame(
            [(file_path, all_files[file_path]["size"], all_files[file_path]["modificationTime"])],
            schema="file_name STRING, file_size LONG, file_modified_time LONG"
        ).withColumn("processed_at", current_timestamp()) \
         .withColumn("status", lit("SUCCESS")) \
         .write.format("delta").mode("append").saveAsTable(WATERMARK_TABLE)

        print(f"Watermark updated: {file_path}")

In [0]:
%sql
-- VALIDATE NYCTAXI.SILVER.REGIONS_SCD_2
SELECT * FROM NYCTAXI.SILVER.REGIONS_SCD_2
ORDER BY ID ASC, effective_end_date DESC

In [0]:
dbutils.notebook.exit('YELLOW TAXI TRIP HAS BEEN LOADED INTO NYCTAXI.SILVER.REGIONS_SCD_2')